In [ ]:
# DeepSeek system prompt for GRPO based training
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)

In [ ]:
!pip install datasets

In [ ]:
from dataclasses import dataclass
from datasets import load_dataset  # 添加这行导入

In [ ]:

# 构造训练数据的函数
def make_conversation(example):
    """
    将数据集中的示例转换为对话格式。
    
    参数:
        example: 数据集中的单个示例
    
    返回:
        转换后的对话格式数据
    """
    """将数据集示例转换为对话格式。"""
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }

In [ ]:
# 加载并准备数据集
def load_math_dataset():
    """加载并准备数学数据集。"""
    dataset = load_dataset(
        "AI-MO/NuminaMath-TIR",
        name="default",
        split=['train', 'test']
    )
    
    # 将拆分转换为字典
    dataset = {
        'train': dataset[0],
        'test': dataset[1]
    }
    
    # 应用对话格式
    for split in dataset:
        dataset[split] = dataset[split].map(make_conversation)

        # 如果存在“messages”列，则删除
        if "messages" in dataset[split].column_names:
            dataset[split] = dataset[split].remove_columns("messages")
    
    return dataset

In [ ]:
#加载我们的训练数据集并打印训练/测试大小
dataset = load_math_dataset()

print(f"Train set size: {len(dataset['train'])}")
print(f"Test set size: {len(dataset['test'])}")

In [ ]:
def validate_dataset(dataset):
    """对数据集执行基本的验证检查。"""
    
    # 定义数据集所需的字段
    required_fields = ["problem", "prompt"]

    # 遍历数据集的 'train' 和 'test' 划分
    for split in ['train', 'test']:
        print(f"\n正在验证 {split} 划分:")

        # 从数据集中获取列名
        fields = dataset[split].column_names

        # 检查是否有任何必需的字段缺失
        missing = [field for field in required_fields if field not in fields]
        if missing:
            print(f"警告: 缺失的字段: {missing}")  # 若字段缺失则发出警告
        else:
            print("✓ 所有必需字段均存在")  # 确认所有字段都存在

        # 从数据集划分中获取第一个样本
        sample = dataset[split][0]

        # 提取 'prompt' 字段，该字段包含一个消息列表
        messages = sample['prompt']

        # 验证提示格式：
        # - 它应该至少包含两条消息
        # - 第一条消息应该来自 'system' 角色
        # - 第二条消息应该来自 'user' 角色
        if (len(messages) >= 2 and
            messages[0]['role'] == 'system' and
            messages[1]['role'] == 'user'):
            print("✓ 提示格式正确")  # 确认格式正确
        else:
            print("警告: 提示格式不正确")  # 若格式不正确则发出警告

# 验证数据集
validate_dataset(dataset)